# BÁO CÁO GIỮA KỲ: Đề xuất và cài đặt phiên bản tuần tự

**Khoa:** Công nghệ thông tin

**Trường:** Đại học Khoa học tự nhiên, ĐHQG-HCM

**GVHD:** thầy Phạm Trọng Nghĩa

**Nhóm:** 07

**Năm học:** 2026 – 2027

---

## 1. Thông tin nhóm và kế hoạch công việc

### 1.1 Danh sách thành viên

| STT | MSSV      | Họ và tên             | Vai trò |
|:---:|-----------|-----------------------|---------|
| 1   | 23120315  | Nguyễn Đăng Pha       |         |
| 2   | 23120356  | Trần Nguyễn Trí Thanh |         |
| 3   | 23120360  | Đặng Lê Đức Thịnh     |         |
| 4   | 23120366  | Trương Thanh Thuận    |         |

### 1.2 Kế hoạch phân công

| Giai đoạn - Công việc                                       | Thành viên phụ trách   | Phiên bản  |
|-------------------------------------------------------------|------------------------|------------|
| Giai đoạn 1 - Tiền xử lý (Preprocessing)                    |                        | V0         |
| Giai đoạn 2 - Phát hiện điểm đặc trưng (Keypoint Detection) |                        | V0         |
| Giai đoạn 3 - Mô tả điểm đặc trưng (Description)            |                        | V0         |
| Giai đoạn 4 - Ghép cặp điểm (Matching)                      |                        | V0         |
| Giai đoạn 5 - Ước lượng Homography                          |                        | V0         |
| Giai đoạn 6 - Biến dạng và hòa trộn (Warping & Blending)    |                        | V0         |
| Đánh giá hiệu năng và báo cáo                               |                        | Tất cả     |

---

## 2. Hiểu vấn đề

### 2.1 Bài toán ghép ảnh toàn cảnh (Panorama Stitching)

Ghép ảnh toàn cảnh là quá trình kết hợp hai (hoặc nhiều) ảnh chụp có vùng chồng lấp của cùng một cảnh vật thành một ảnh góc rộng duy nhất...

**Đầu vào:** Hai ảnh RGB có vùng chồng lấp (ảnh A và ảnh B).  
**Đầu ra:** Một ảnh toàn cảnh duy nhất (ảnh B được biến dạng và hòa vào khung của ảnh A).

### 2.2 Pipeline 6 giai đoạn

| #   | Giai đoạn                    | Mô tả                                                                         |
|:---:|------------------------------|-------------------------------------------------------------------------------|
| 1   | **Tiền xử lý**               | Chuyển ảnh sang ảnh xám (grayscale) và làm mịn để giảm nhiễu                  |
| 2   | **Phát hiện điểm đặc trưng** | Tìm các điểm nổi bật (góc cạnh) trong từng ảnh                                |
| 3   | **Mô tả điểm đặc trưng**     | Tính vector mô tả nhỏ gọn cho vùng lân cận mỗi điểm                           |
| 4   | **Ghép cặp điểm**            | Với mỗi điểm ở ảnh A, tìm điểm tương đồng nhất ở ảnh B; loại bỏ các cặp mơ hồ |
| 5   | **Ước lượng Homography**     | Tính ma trận biến đổi xạ ảnh 3×3 để ánh xạ ảnh B sang ảnh A, xử lý nhiễu      |
| 6   | **Biến dạng và hòa trộn**    | Biến dạng ảnh B vào hệ tọa độ của ảnh A và hòa trộn vùng chồng lấp            |

---

## 3. Phương pháp được chọn

Nhóm chọn phương pháp cho từng giai đoạn dựa trên hai tiêu chí: **chất lượng đầu ra** và **khả năng tăng tốc trên GPU**.

<table>
  <thead>
    <tr>
      <th>Giai đoạn</th>
      <th>Phương pháp lựa chọn</th>
      <th>Cơ chế hoạt động</th>
      <th>Tham số chính</th>
      <th>Ưu thế chất lượng</th>
      <th>Tính phù hợp với GPU</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>1. Tiền xử lý</td>
      <td>Grayscale cố định + Separable Gaussian Blur (1D Horiz → 1D Vert)</td>
      <td>Chuyển RGB→Gray bằng công thức BT.601; nhân chập 1D theo hàng rồi theo cột thay vì kernel 2D đầy đủ</td>
      <td>σ (độ lệch chuẩn), k = 2⌈3σ⌉+1</td>
      <td>Loại bỏ nhiễu tần cao hiệu quả; O(2kN) thay vì O(k²N) mỗi pixel</td>
      <td>Mỗi pixel độc lập, hai pass 1D dễ map 1 thread/pixel; cache-friendly hơn kernel 2D</td>
    </tr>
    <tr>
      <td>2. Phát hiện điểm đặc trưng</td>
      <td>Harris Corner + NMS + Grid-based Top-K</td>
      <td>Tính gradient Ix/Iy bằng Sobel 3×3; làm mượt structure tensor; tính R = det(M) − k·trace(M)²; threshold, NMS rồi giữ Top-K theo từng ô lưới</td>
      <td>k = 0.04, σ_w = 1.0–1.5, R_thresh = 0.01·max(R), NMS 3×3, grid 32×32</td>
      <td>Response liên tục để xếp hạng; NMS loại cụm điểm; Grid Top-K giúp keypoint phân bố đều cho matching và homography</td>
      <td>Các bước là stencil/pixel-wise phù hợp 1 thread/pixel; có thể tối ưu bằng shared-memory tiling, reduction và parallel compaction</td>
    </tr>
    <tr>
      <td>3. Mô tả điểm đặc trưng</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>4. Ghép cặp điểm</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>5. Ước lượng Homography</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
    <tr>
      <td>6. Biến dạng và hòa trộn</td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
      <td></td>
    </tr>
  </tbody>
</table>


### 3.1 Giai đoạn 1 – Tiền xử lý: Grayscale cố định + Separable Gaussian Blur

#### Cơ chế hoạt động

**Bước 1 – RGB → Grayscale (công thức cố định BT.601):**

```
Gray = 0.299·R + 0.587·G + 0.114·B
```

Trọng số cố định → mỗi pixel RGB được xử lý **độc lập hoàn toàn**, không có sự phụ thuộc dữ liệu giữa các pixel.

**Bước 2 – Separable Gaussian Blur (tách kernel 2D thành 2 lần nhân chập 1D):**

Tận dụng tính chất phân ly (separability) của kernel Gaussian:

```
G(x,y) = g(x) · g(y)  →  Pass 1: nhân chập 1D theo hàng
                         Pass 2: nhân chập 1D theo cột
```

Độ phức tạp giảm từ **O(k²·N)** xuống **O(2k·N)** với N là tổng số pixel và k là kích thước kernel.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| σ | Độ lệch chuẩn Gaussian, kiểm soát mức độ làm mịn | 1.0 – 1.6 |
| k | Kích thước kernel, k = 2⌈3σ⌉+1 | 7 (với σ = 1.0) |

#### Lý do phù hợp với song song hóa

- **Grayscale:** Mỗi pixel độc lập → map trực tiếp `1 thread = 1 pixel`, không cần đồng bộ.
- **Pass 1D ngang:** Mỗi hàng (row) độc lập → song song hóa theo hàng.
- **Pass 1D dọc:** Mỗi cột (column) độc lập → tương tự.
- **Cache-friendly:** Đọc bộ nhớ tuần tự 1D thay vì nhảy cách 2D → giảm cache miss đáng kể.


### 3.2 Giai đoạn 2 – Harris Corner + NMS + Grid-based Top-K

Nhóm chốt **Harris Corner** làm detector duy nhất cho V0–V3 để bảo đảm phép đo speedup so sánh cùng thuật toán và tham số. 

#### Cơ chế hoạt động

**Bước 1 – Tính gradient:** Dùng Sobel 3×3 để tính mức biến thiên cường độ theo hai hướng `Ix` và `Iy` tại mỗi pixel.

**Bước 2 – Xây dựng structure tensor:** Tính ba tích `Ix²`, `Iy²`, `IxIy`, sau đó làm mượt bằng Gaussian để thu `Sxx`, `Syy`, `Sxy`. Các giá trị này tạo ma trận mô tả mức biến thiên cường độ theo hai hướng:

```
M = [[Sxx, Sxy], [Sxy, Syy]]
```

**Bước 3 – Tính Harris response:**

```
R = det(M) - k·trace(M)²
```

`R` lớn và dương biểu thị ứng viên góc; `R` âm thường là cạnh; `R` gần 0 là vùng phẳng.

**Bước 4 – Lọc và chọn keypoint:** Giữ các điểm thỏa `R > α·max(R)`, áp dụng NMS để chỉ giữ cực đại cục bộ, rồi dùng Grid-based Top-K để giới hạn số lượng và phân bố keypoint đều trên ảnh.

#### Tham số chính

| Tham số | Ý nghĩa | Giá trị đề xuất |
|---------|---------|------------------|
| `k` | Độ nhạy của Harris response với cạnh | 0.04 |
| `σ_w` | Độ lệch chuẩn Gaussian khi làm mượt structure tensor | 1.0 – 1.5 |
| Tensor window | Kích thước cửa sổ làm mượt tensor | 3×3 hoặc 5×5 |
| `α` | Hệ số threshold tương đối trong `R > α·max(R)` | 0.01 |
| NMS window | Cửa sổ giữ cực đại cục bộ | 3×3 |
| Grid cell | Kích thước ô dùng cho Spatial Top-K | 32×32 pixel |
| K/cell | Số keypoint tối đa giữ lại trong mỗi ô | Chốt bằng thực nghiệm V0 |

#### Lý do phù hợp với song song hóa

- **Sobel, tensor và Harris response:** Mỗi pixel được ánh xạ trực tiếp thành `1 thread = 1 pixel`; luồng điều khiển đều, ít phân kỳ warp.
- **Stencil cục bộ:** Các pixel lân cận được tái sử dụng bằng shared-memory tiling có halo, giúp giảm số lần đọc global memory.
- **Threshold và NMS:** Mỗi thread kiểm tra response của pixel mình trên response map đã hoàn tất, không có phụ thuộc ghi giữa các pixel.
- **Tìm `max(R)` và thu thập keypoint:** Có thể dùng các mẫu song song chuẩn như reduction, prefix-sum compaction hoặc warp-level aggregation.
- **Grid-based Top-K:** Mỗi ô ảnh có thể được một CUDA block xử lý độc lập, vừa song song hóa chọn điểm vừa bảo đảm phân bố không gian.
- **Khả năng tối ưu theo phiên bản:** V1 dùng global memory; V2 dùng shared-memory tiling; V3 tối ưu compaction và Spatial Top-K dựa trên bottleneck đo được.

Các tham số được chốt sau thí nghiệm V0 và giữ nguyên trong V1–V3.


---

## 4. Phiên bản tuần tự (V0)

Phiên bản V0 là cài đặt C/C++ **đơn luồng** (không dùng GPU) của toàn bộ pipeline ghép ảnh.

---

## 5. Kết quả và phân tích hiệu năng V0

---

### 5.1 Độ chính xác

### 5.2 Ảnh toàn cảnh đầu ra

### 5.3 Bảng thời gian thực thi theo từng giai đoạn

### 5.4 Phân tích giai đoạn chiếm nhiều thời gian nhất

### 5.5 Ước lượng theo Định luật Amdahl

---

## 6. Bộ dữ liệu

### 6.1 Quy chuẩn ảnh và cách chọn cặp ảnh

Mỗi cặp ảnh được **cắt tỉ lệ 16:9** và resize về 3 độ phân giải:

| Tên | Độ phân giải | Mục đích                          |
|-----|--------------|-----------------------------------|
| Nhỏ | 1280 × 720   | Phát triển nhanh và debug         |
| Vừa | 1920 × 1080  | Benchmark chính                   |
| Lớn | 3840 × 2160  | Kiểm tra khả năng mở rộng của GPU |

### 6.2 Danh sách địa điểm chụp

| STT | Địa điểm    | Bối cảnh      | Phân loại   | Mô tả cặp ảnh   | Đường dẫn |
|-----|-------------|---------------|-------------|-----------------|-----------|
| 1   |             | Indoor        | Standard    |                 |           |
| 2   |             | Outdoor       | Standard    |                 |           |
| 3   |             | Team's choice | Standard    |                 |           |
| 4   |             | Team's choice | Challenging |                 |           |

**Link bộ dữ liệu (Google Drive):** ...

---

## 7. Kế hoạch GPU

### 7.1 Các giai đoạn sẽ chuyển lên GPU

### 7.2 Chi tiết ánh xạ luồng V1

### 7.3 Ý tưởng tối ưu hóa cho V2, V3, ...

1. **[Ý tưởng 1]**

2. **[Ý tưởng 2]**

3. **[Ý tưởng 3]**

---

## Tài liệu tham khảo